In [3]:
# ================================================================
# TORCH: Factory-Level Risk Index — Myanmar Apparel Supply Chains
# Stage 2: Text Preprocessing
# ================================================================

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources (only needed once)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

news = pd.read_csv("stage1_news.csv")
print("Articles loaded:", len(news))

# ----------------------------------------------------------------
# 1. Remove articles that failed to scrape
#    Some articles returned server error messages instead of
#    content during scraping. We drop these rows entirely.
# ----------------------------------------------------------------

error_phrases = [
    "proxy server received an invalid response",
    "server is temporarily unable to service",
    "failed to scrape"
]

def is_error(text):
    if pd.isna(text):
        return True
    return any(phrase in str(text).lower() for phrase in error_phrases)

news = news[~news["Content"].apply(is_error)].reset_index(drop=True)
print("Articles after removing scrape errors:", len(news))

# ----------------------------------------------------------------
# 2. Remove the boilerplate footer
#    Every article ends with the same two sentences about
#    "uphold the rule of law" — this is site boilerplate,
#    not complaint content, so we strip it out.
# ----------------------------------------------------------------

FOOTER = (
    "To uphold the rule of law needed to improve the social-economic life "
    "of the working class in Myanmar. To help the transition to democracy "
    "as media role in the workforce gender equity."
)

news["content_clean"] = news["Content"].str.replace(FOOTER, "", regex=False).str.strip()

# ----------------------------------------------------------------
# 3. Basic text cleaning
#    content_clean keeps numbers — used for extracting kyats
#    amounts and worker counts in the dashboard.
#    content_for_nlp removes numbers — used for tokenization
#    and ML model input.
# ----------------------------------------------------------------

def basic_clean(text, remove_numbers=False):
    if pd.isna(text):
        return ""
    text = text.lower()
    if remove_numbers:
        text = re.sub(r"\d+", "", text)      # remove numbers only for NLP tokens
    text = re.sub(r"[^\w\s]", "", text)      # remove punctuation
    text = re.sub(r"\s+", " ", text).strip() # collapse whitespace
    return text

news["content_clean"]   = news["content_clean"].apply(lambda t: basic_clean(t, remove_numbers=False))
news["content_for_nlp"] = news["content_clean"].apply(lambda t: basic_clean(t, remove_numbers=True))

# ----------------------------------------------------------------
# 4. Tokenize, remove stopwords, and lemmatize
#    - Tokenize: split text into individual words (tokens)
#    - Stopwords: remove common words like "the", "is", "and"
#      that carry no meaningful signal for risk classification
#    - Lemmatize: reduce words to their base form
#      e.g. "dismissed" → "dismiss", "forcing" → "force"
# ----------------------------------------------------------------

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Domain-specific stopwords that appear in almost every article
# but carry no risk signal on their own
custom_stopwords = {
    "worker", "workers", "factory", "said", "also", "would",
    "told", "according", "myanmar", "garment", "one", "two",
    "three", "four", "five", "six", "seven", "eight", "nine", "ten"
}
stop_words.update(custom_stopwords)

def preprocess(text):
    tokens = word_tokenize(text)                           # split into words
    tokens = [t for t in tokens if t not in stop_words]   # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]    # lemmatize
    tokens = [t for t in tokens if len(t) > 2]            # drop very short tokens
    return tokens

news["tokens"]            = news["content_for_nlp"].apply(preprocess)
news["content_processed"] = news["tokens"].apply(lambda t: " ".join(t))

# ----------------------------------------------------------------
# 5. Sanity check — print a sample before and after
# ----------------------------------------------------------------

print("\nOriginal content (first 300 chars):")
print(news.loc[0, "Content"][:300])
print("\nProcessed content (first 300 chars):")
print(news.loc[0, "content_processed"][:300])
print("\nToken count (first article):", len(news.loc[0, "tokens"]))

# ----------------------------------------------------------------
# 6. Dataset-level stats
# ----------------------------------------------------------------

news["token_count"] = news["tokens"].apply(len)

print("\nToken count summary:")
print(news["token_count"].describe().round(1))

empty = (news["token_count"] == 0).sum()
print(f"\nArticles with no tokens after preprocessing: {empty}")
if empty > 0:
    print(news[news["token_count"] == 0][["title", "URL"]])

# ----------------------------------------------------------------
# 7. Save output for Stage 3
# ----------------------------------------------------------------

news.to_csv("stage2_news_preprocessed.csv", index=False, encoding="utf-8-sig")
print("\nStage 2 done. Saved to stage2_news_preprocessed.csv")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/macbookpro/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/macbookpro/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/macbookpro/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/macbookpro/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Articles loaded: 1441
Articles after removing scrape errors: 1413

Original content (first 300 chars):
Workers at the Chai Moon Sports (Myanmar) garment factory, located in Ward No. 13, Pathein Township, Ayeyarwady Region, say that those who refuse to work overtime on factory holidays are required to sign warning letters, and some workers have been dismissed. “Lines 1, 2, 5, 6, and 7 are called most 

Processed content (first 300 chars):
chai moon sport located ward pathein township ayeyarwady region say refuse work overtime holiday required sign warning letter dismissed line called frequently overtime sign warning letter dont work overtime holiday day production urgent call overnight work give rest make continue working next mornin

Token count (first article): 160

Token count summary:
count    1413.0
mean      179.1
std        72.0
min         0.0
25%       134.0
50%       171.0
75%       214.0
max       775.0
Name: token_count, dtype: float64

Articles with no tokens after preproce

In [4]:
import pandas as pd

news = pd.read_csv("stage2_news_preprocessed.csv")
print("Total articles:", len(news))
print("\nToken count summary:")
print(news["token_count"].describe().round(1))

Total articles: 1410

Token count summary:
count    1410.0
mean      179.4
std        71.6
min         0.0
25%       134.0
50%       171.0
75%       214.0
max       775.0
Name: token_count, dtype: float64


In [5]:
print(news[news["token_count"] == 0][["title", "URL"]])

                                                  title  \
1152  Hanboom (or) Handa-3 Garment အချိန်ပိုလုပ်ခ မပ...   

                                                    URL  
1152  https://www.myanmarlabournews.com/en/posts/han...  


In [6]:
news = news[news["token_count"] > 0].reset_index(drop=True)
news.to_csv("stage2_news_preprocessed.csv", index=False, encoding="utf-8-sig")
print("Articles remaining:", len(news))

Articles remaining: 1409
